In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# Ensure timestamp columns are numeric
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce')

# Timezone conversion helper (UTC -> Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
def ts_to_shanghai(ts):
    return pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) metric.csv summaries
df_metric_cmdb = df_metric['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='rows')
df_metric_kpi_counts = df_metric['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='rows').head(20)
metric_ts_range = pd.DataFrame({
    'metric_ts_min': [ts_to_shanghai(df_metric['timestamp'].min())],
    'metric_ts_max': [ts_to_shanghai(df_metric['timestamp'].max())]
})

# 2) trace.csv summaries
df_trace_cmdb = df_trace['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='rows')
df_trace_trace_counts = df_trace['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='rows').head(20)
trace_ts_range = pd.DataFrame({
    'trace_ts_min': [ts_to_shanghai(df_trace['timestamp'].min())],
    'trace_ts_max': [ts_to_shanghai(df_trace['timestamp'].max())]
})

# 3) metric.csv group stats (count, P50, P90, P95, P99) top 20 by count
metric_group = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda s: s.quantile(0.5),
         p90=lambda s: s.quantile(0.9),
         p95=lambda s: s.quantile(0.95),
         p99=lambda s: s.quantile(0.99))
)
metric_group[['p50','p90','p95','p99']] = metric_group[['p50','p90','p95','p99']].round(6)
metric_group_top20 = metric_group.sort_values('count', ascending=False).head(20)

# 4) trace.csv group stats (count, P50, P90, P95, P99) top 20 by count
trace_group = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count',
         p50=lambda s: s.quantile(0.5),
         p90=lambda s: s.quantile(0.9),
         p95=lambda s: s.quantile(0.95),
         p99=lambda s: s.quantile(0.99))
)
trace_group[['p50','p90','p95','p99']] = trace_group[['p50','p90','p95','p99']].round(6)
trace_group_top20 = trace_group.sort_values('count', ascending=False).head(20)

# Display compact results (CSV-like tables). Using variables (IPython) for output.
df_metric_cmdb, df_metric_kpi_counts, metric_ts_range, metric_group_top20, df_trace_cmdb, df_trace_trace_counts, trace_ts_range, trace_group_top20

```
Out[1]:
```
```python
# Use existing variables from prior execution to compose a plain-English summary string and display it.
# The summary is compact and based on the DataFrame results already computed:
metric_cmdb_count = int(df_metric_cmdb.shape[0])
metric_cmdb_examples = ", ".join(df_metric_cmdb['cmdb_id'].head(8).tolist())
metric_kpi_unique = int(df_metric['kpi_name'].nunique())
metric_kpi_top20_csv = df_metric_kpi_counts.to_csv(index=False)
metric_ts_min = metric_ts_range.loc[0, 'metric_ts_min']
metric_ts_max = metric_ts_range.loc[0, 'metric_ts_max']

trace_cmdb_count = int(df_trace_cmdb.shape[0])
trace_cmdb_examples = ", ".join(df_trace_cmdb['cmdb_id'].head(8).tolist())
trace_name_unique = int(df_trace['trace_name'].nunique())
trace_name_top20_csv = df_trace_trace_counts.to_csv(index=False)
trace_ts_min = trace_ts_range.loc[0, 'trace_ts_min']
trace_ts_max = trace_ts_range.loc[0, 'trace_ts_max']

# Top groups (show up to 5 rows each, CSV-like)
metric_groups_top5_csv = metric_group_top20.head(5).to_csv(index=False)
trace_groups_top5_csv = trace_group_top20.head(5).to_csv(index=False)

summary = f"""
Metric.csv summary:
- Unique cmdb_id count: {metric_cmdb_count} (examples: {metric_cmdb_examples}, ...)
- Unique kpi_name count: {metric_kpi_unique}
- File timestamp range (Asia/Shanghai): {metric_ts_min}  ->  {metric_ts_max}
- Top 20 kpi_name by row counts (CSV):
{metric_kpi_top20_csv.strip()}

- Top groups by (cmdb_id, kpi_name) (top 5 shown; columns: cmdb_id,kpi_name,count,p50,p90,p95,p99):
{metric_groups_top5_csv.strip()}


Trace.csv summary:
- Unique cmdb_id count: {trace_cmdb_count} (examples: {trace_cmdb_examples}, ...)
- Unique trace_name count: {trace_name_unique}
- File timestamp range (Asia/Shanghai): {trace_ts_min}  ->  {trace_ts_max}
- Top 20 trace_name by row counts (CSV):
{trace_name_top20_csv.strip()}

- Top groups by (cmdb_id, trace_name) (top 5 shown; columns: cmdb_id,trace_name,count,p50,p90,p95,p99):
{trace_groups_top5_csv.strip()}
""".strip()

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id  rows
0       db_001  1648
1       db_010  1647
2       db_008  1646
3       db_003  1646
4       db_011  1646
..         ...   ...
51  docker_008   432
52  docker_005   423
53  docker_007   423
54  docker_006   396
55     osb_001   244

[56 rows x 2 columns],                     kpi_name  rows
0               Disk_io_util  1063
1                 Send_total  1056
2                 Recv_total  1053
3            Memory_used_pct  1053
4                    Page_po  1052
5                Memory_used  1052
6                    Page_pi  1052
7           Memory_available  1052
8                Disk_rd_ios  1051
9                Disk_wr_ios  1050
10               Disk_rd_kbs  1050
11                Disk_svctm  1050
12             Disk_avgqu_sz  1050
13                Agent_ping  1050
14           CPU_system_time  1050
15              CPU_idle_pct  1050
16                 ICMP_ping  1050
17           CPU_iowait_time  1050
18                Disk_await  1050
19  Outgoing_network_traffic  1049,              metric_ts_min            metric_ts_max
0  2020-05-27 01:30:00 CST  2020-05-27 02:30:00 CST,       cmdb_id                  kpi_name  count         p50         p90         p95         p99
1779  osb_001           app.succee_rate     61    1.000000    1.000000    1.000000    1.000000
1778  osb_001            app.succee_num     61  525.000000  562.000000  574.000000  599.600000
1777  osb_001                   app.num     61  533.000000  562.000000  574.000000  599.600000
1776  osb_001              app.avg_time     61    0.000528    0.000603    0.000630    0.000691
1390   os_015  Incoming_network_traffic     50    0.002284    0.002344    0.002374    0.005180
288    db_007             MEM_real_util     49   80.330000   80.340000   80.340000   80.365600
95     db_003              CPU_free_pct     49   97.666648   98.834076   98.975657   99.251117
1073   os_009              Disk_io_util     49    0.000000    0.000000    0.000000    0.000000
1251   os_012      Processor_load_5_min     49    0.010000    0.020000    0.023333    0.025933
1053   os_008                Send_total     49    0.000560    0.000676    0.000679    0.000680
953    os_006                Send_total     49    0.000367    0.000466    0.000470    0.000472
1633   os_020              Disk_io_util     49    0.000000    0.000000    0.204446    0.760000
1276   os_013              Disk_io_util     49    0.000000    0.100000    0.500000    0.508368
922    os_006              Disk_io_util     49    0.000000    0.000000    0.000000    0.000000
320    db_008              CPU_free_pct     49   97.970758   98.572953   98.679943   98.782248
1511   os_017                Send_total     49    0.256156    0.280210    0.288241    0.306988
974    os_007               Disk_rd_ios     49    0.000000    0.000000    0.000000    0.000000
1582   os_019              Disk_io_util     49    0.000000    0.000000    0.000000    0.512459
1225   os_012              Disk_io_util     49    0.508328    1.000000    1.600000    2.000000
1023   os_008              Disk_io_util     49    0.000000    0.000000    0.000000    0.000000,       cmdb_id  rows
0  docker_001   840
1  docker_002   840
2  docker_003   840
3  docker_004   840
4  docker_007   532
5  docker_008   528
6  docker_006   480
7  docker_005   480
8      os_021   480
9      os_022   480,                                        trace_name  rows
0                  trace.self.local.duration_mean   240
1                   trace.self.local.duration_p95   240
2                     trace.self.local.error_rate   240
3                      trace.self.local.row_count   240
4                       trace.self.jdbc.row_count   240
5                      trace.self.jdbc.error_rate   240
6                    trace.self.jdbc.duration_p95   240
7                   trace.self.jdbc.duration_mean   240
8                        trace.self.csf.row_count   180
9                       trace.self.csf.error_rate   180
10                    trace.self.csf.duration_p95   180
11                   trace.self.csf.duration_mean   180
12                 trace.self.flyremote.row_count   120
13                trace.self.flyremote.error_rate   120
14              trace.self.flyremote.duration_p95   120
15             trace.self.flyremote.duration_mean   120
16      trace.from_os_022.remoteprocess.row_count    60
17     trace.from_os_022.remoteprocess.error_rate    60
18   trace.from_os_022.remoteprocess.duration_p95    60
19  trace.from_os_022.remoteprocess.duration_mean    60,               trace_ts_min             trace_ts_max
0  2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST,        cmdb_id                                     trace_name  count          p50          p90          p95          p99
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30     0.347807     0.484286     0.519370     0.644529
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30     0.867925     1.403285     1.566297     2.429595
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30     0.000000     0.000000     0.000000     0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count     30   129.000000   144.000000   145.650000   152.680000
4   docker_001                   trace.self.csf.duration_mean     30     0.040913     0.051028     0.053892     0.057281
5   docker_001                    trace.self.csf.duration_p95     30     0.098650     0.120205     0.163455     0.191795
6   docker_001                      trace.self.csf.error_rate     30     0.000000     0.000000     0.000000     0.000000
7   docker_001                       trace.self.csf.row_count     30   516.000000   576.000000   582.600000   610.720000
8   docker_001             trace.self.flyremote.duration_mean     30     0.006981     0.009119     0.009803     0.010533
9   docker_001              trace.self.flyremote.duration_p95     30     0.007525     0.025545     0.056815     0.073949
10  docker_001                trace.self.flyremote.error_rate     30     0.000000     0.000000     0.000000     0.000000
11  docker_001                 trace.self.flyremote.row_count     30   129.000000   144.000000   145.650000   152.680000
12  docker_001                  trace.self.jdbc.duration_mean     30     0.002324     0.003184     0.003468     0.003618
13  docker_001                   trace.self.jdbc.duration_p95     30     0.003000     0.003000     0.003000     0.011520
14  docker_001                     trace.self.jdbc.error_rate     30     0.000000     0.000000     0.000000     0.002247
15  docker_001                      trace.self.jdbc.row_count     30  2033.500000  2304.200000  2331.300000  2447.140000
16  docker_001                 trace.self.local.duration_mean     30     0.007106     0.011159     0.011995     0.014247
17  docker_001                  trace.self.local.duration_p95     30     0.057725     0.086300     0.089550     0.094260
18  docker_001                    trace.self.local.error_rate     30     0.000000     0.071429     0.071429     0.071429
19  docker_001                     trace.self.local.row_count     30  1806.000000  2016.000000  2039.100000  2137.520000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# 1) Global thresholds (P95 and P5) and global counts per (cmdb_id, kpi_name)
thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         p95=lambda s: s.quantile(0.95),
         p5=lambda s: s.quantile(0.05))
)
thresh[['p95','p5']] = thresh[['p95','p5']].round(6)

# 2) Filter metric.csv to incident window (Asia/Shanghai 2020-05-27 02:00:00 through 02:30:00 inclusive)
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window and detect anomalies: value >= p95 OR value <= p5
merged = df_window.merge(thresh, on=['cmdb_id','kpi_name'], how='left')
merged['is_anom'] = (merged['value'] >= merged['p95']) | (merged['value'] <= merged['p5'])

# Aggregations per group within window
window_stats = (
    merged
    .groupby(['cmdb_id','kpi_name'], as_index=True)
    .agg(window_total_rows=('value','count'),
         max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
)

anom_stats = (
    merged[merged['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=True)
    .agg(anomaly_count=('is_anom','sum'),
         earliest_anom_ts_num=('timestamp','min'),
         latest_anom_ts_num=('timestamp','max'))
)

# Combine with global thresholds and counts
result = (
    thresh.set_index(['cmdb_id','kpi_name'])
    .join(window_stats, how='left')
    .join(anom_stats, how='left')
    .reset_index()
)

# Keep only groups with at least one anomaly in the window
result = result[result['anomaly_count'].fillna(0) >= 1].copy()

# Fill NA and round numeric cols
result['window_total_rows'] = result['window_total_rows'].fillna(0).astype(int)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].round(6)
result['min_value_in_window'] = result['min_value_in_window'].round(6)
result['p95'] = result['p95'].round(6)
result['p5']  = result['p5'].round(6)

# Prepare formatted timestamp strings for display, but keep numeric for sorting
def fmt_ts_num(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_anomaly_timestamp'] = result['earliest_anom_ts_num'].apply(fmt_ts_num)
result['latest_anomaly_timestamp'] = result['latest_anom_ts_num'].apply(fmt_ts_num)

# Select and order columns as requested
result = result[[
    'cmdb_id','kpi_name','global_count','p95','p5',
    'window_total_rows','anomaly_count','earliest_anomaly_timestamp',
    'latest_anomaly_timestamp','max_value_in_window','min_value_in_window',
    'earliest_anom_ts_num'  # keep numeric helper for sorting
]]

# 4) Sort and limit to top 20 rows (anomaly_count desc, earliest_anom_ts_num asc)
result_sorted = result.sort_values(by=['anomaly_count','earliest_anom_ts_num'], ascending=[False, True]).head(20)

# Drop helper numeric column for final display
result_top20 = result_sorted.drop(columns=['earliest_anom_ts_num'])

result_top20

```
Out[2]:
```
Summary (plain English):

- I analyzed metric.csv for the incident window 2020-05-27 02:00:00–02:30:00 CST and report groups (cmdb_id, kpi_name) that had at least one anomaly versus global thresholds (global P95 and P5 computed from the full series).
- Top 20 groups (shown) all have very high anomaly counts: most groups show anomaly_count equal to the window size (30 or 31), meaning every sampled point in the window was flagged as anomalous against the global P95/P5.
- Examples from the top results:
  - os_007 / Disk_rd_ios — global_count 49, p95=0.0, p5=0.0, window_total_rows=31, anomaly_count=31, max=0.0, min=0.0 (all points equal to thresholds).
  - osb_001 / app.succee_rate — global_count 61, p95=1.0, p5=0.0, window_total_rows=31, anomaly_count=31, max=1.0, min=0.0.
  - db_001 / ACS — global_count 47, p95=0.02, p5=0.01, window_total_rows=30, anomaly_count=30, max=0.02, min=0.0.
  - db_001 / AIOS, Hang, Login_Per_Sec, MEM_Total, On_Off_State, Physical_Read_Per_Sec, Proc_Used_Pct, Row_Lock, SctRead_Per_Sec, tnsping_result_time — each shows 30/30 anomalies in the window (global counts ~47–48).
  - Similar full-window anomaly patterns appear for db_002, db_003 metrics in the top 20.
- Interpretation: Many metrics are constant or at extreme quantiles (p5 or p95 equal to observed values), so the global thresholds mark nearly every point in the incident window as anomalous. This often indicates either (a) a real, persistent deviation during the window, or (b) thresholds coinciding with constant/quantized metric values leading to every point being flagged.
- Next steps recommended: inspect a few representative raw time series around the window (to confirm real deviation vs threshold artefact) and prioritize metrics where max/min show meaningful deviations (e.g., tnsping_result_time had max=10 and min=0) for further troubleshooting.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id               kpi_name  global_count         p95          p5  window_total_rows  anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  min_value_in_window
974    os_007            Disk_rd_ios            49    0.000000    0.000000                 31             31    2020-05-27 02:00:00 CST  2020-05-27 02:30:00 CST             0.000000             0.000000
1779  osb_001        app.succee_rate            61    1.000000    0.000000                 31             31    2020-05-27 02:00:00 CST  2020-05-27 02:30:00 CST             1.000000             0.000000
0      db_001                    ACS            47    0.020000    0.010000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.020000             0.000000
1      db_001                   AIOS            47    0.010000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.010000             0.000000
10     db_001                   Hang            47    1.000000    1.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.000000             1.000000
14     db_001          Login_Per_Sec            47    0.400000    0.300000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.400000             0.300000
15     db_001              MEM_Total            48  381.902264  381.902264                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           381.902264           381.902264
21     db_001           On_Off_State            47    1.000000    1.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.000000             1.000000
24     db_001  Physical_Read_Per_Sec            47  313.370000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           324.000000             0.000000
25     db_001          Proc_Used_Pct            47    1.540000    1.530000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.540000             1.530000
26     db_001     Proc_User_Used_Pct            47    1.540000    1.530000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.540000             1.530000
28     db_001               Row_Lock            47    0.000000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.000000             0.000000
29     db_001        SctRead_Per_Sec            47    0.000000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.000000             0.000000
44     db_001    tnsping_result_time            48   10.000000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST            10.000000             0.000000
45     db_002                    ACS            47    0.020000    0.013000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.020000             0.010000
46     db_002                   AIOS            47    0.000000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.010000             0.000000
60     db_002              MEM_Total            48  381.902264  381.902264                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           381.902264           381.902264
89     db_002    tnsping_result_time            48   10.000000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST            10.000000             0.000000
92     db_003                    AWS            47    0.020000    0.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             0.020000             0.000000
100    db_003                   Hang            47    1.000000    1.000000                 30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.000000             1.000000```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing dataframes if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# Incident window in UTC corresponding to 2020-05-27 02:00:00 - 2020-05-27 02:30:00 Asia/Shanghai
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 1) Trace groups of interest
trace_groups = [
    ('docker_001', 'trace.from_os_022.remoteprocess.duration_mean'),
    ('docker_001', 'trace.from_os_022.remoteprocess.duration_p95'),
    ('docker_001', 'trace.self.local.duration_mean'),
    ('docker_001', 'trace.self.local.duration_p95'),
]

# Compute global P95 and counts for all trace groups (global thresholds computed BEFORE filtering)
trace_thresh = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95))
)
trace_thresh['global_p95'] = trace_thresh['global_p95'].round(6)

# Filter to selected groups' thresholds
trace_sel_thresh = trace_thresh.merge(
    pd.DataFrame(trace_groups, columns=['cmdb_id','trace_name']),
    on=['cmdb_id','trace_name'],
    how='right'
)

# Filter trace to window first (per rules thresholds are already global)
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge window data with thresholds for selected groups
merged_trace = df_trace_window.merge(trace_sel_thresh, on=['cmdb_id','trace_name'], how='inner')

# Mark anomalies where value >= global_p95
merged_trace['is_anom'] = merged_trace['value'] >= merged_trace['global_p95']

# Aggregate per group in selection: only groups with >=1 anomaly will be returned
trace_agg_window = (
    merged_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(window_row_count=('value','count'),
         anomaly_count_in_window=('is_anom','sum'),
         earliest_anomaly_ts_num=('timestamp', lambda s: s[merged_trace.loc[s.index,'is_anom']].min() if len(s)>0 else pd.NA),
         latest_anomaly_ts_num=('timestamp', lambda s: s[merged_trace.loc[s.index,'is_anom']].max() if len(s)>0 else pd.NA),
         max_value_in_window=('value','max'))
)

# Join global counts/p95
trace_results = trace_sel_thresh.merge(trace_agg_window, on=['cmdb_id','trace_name'], how='left')

# Keep only groups with anomaly_count_in_window >=1
trace_results = trace_results[trace_results['anomaly_count_in_window'].fillna(0) >= 1].copy()

# Format timestamps and tidy columns
def fmt_ts(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

trace_results['earliest_anomaly_timestamp'] = trace_results['earliest_anomaly_ts_num'].apply(fmt_ts)
trace_results['latest_anomaly_timestamp'] = trace_results['latest_anomaly_ts_num'].apply(fmt_ts)

trace_results = trace_results[[
    'cmdb_id','trace_name','global_count','global_p95',
    'window_row_count','anomaly_count_in_window',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window'
]]

trace_results = trace_results.sort_values(by=['anomaly_count_in_window','earliest_anomaly_timestamp'], ascending=[False,True]).head(20)
trace_results[['global_p95','max_value_in_window']] = trace_results[['global_p95','max_value_in_window']].round(6)


# 2) metric.csv: tnsping_result_time for db_001, db_002, db_003
dbs = ['db_001','db_002','db_003']
kpi = 'tnsping_result_time'

# Compute global P95 and counts BEFORE filtering
metric_thresh = (
    df_metric[df_metric['kpi_name']==kpi]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95))
)
metric_thresh['global_p95'] = metric_thresh['global_p95'].round(6)

# Filter to our dbs of interest
metric_sel_thresh = metric_thresh[metric_thresh['cmdb_id'].isin(dbs)].copy()

# Filter metric to window
df_metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge window with thresholds
merged_metric = df_metric_window.merge(metric_sel_thresh, on=['cmdb_id','kpi_name'], how='inner')

# Define anomaly: value >= global_p95 OR (if global_p95 == 0 then value > 0)
merged_metric['is_anom'] = merged_metric.apply(
    lambda r: (r['value'] >= r['global_p95']) if r['global_p95'] > 0 else (r['value'] > 0),
    axis=1
)

# Aggregate per db
metric_agg_window = (
    merged_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(window_row_count=('value','count'),
         anomaly_count_in_window=('is_anom','sum'),
         earliest_anomaly_ts_num=('timestamp', lambda s: s[merged_metric.loc[s.index,'is_anom']].min() if len(s)>0 else pd.NA),
         latest_anomaly_ts_num=('timestamp', lambda s: s[merged_metric.loc[s.index,'is_anom']].max() if len(s)>0 else pd.NA),
         max_value_in_window=('value','max'))
)

metric_results = metric_sel_thresh.merge(metric_agg_window, on=['cmdb_id','kpi_name'], how='left')

# Keep only groups with anomaly_count_in_window >=1
metric_results = metric_results[metric_results['anomaly_count_in_window'].fillna(0) >= 1].copy()

metric_results['earliest_anomaly_timestamp'] = metric_results['earliest_anomaly_ts_num'].apply(lambda x: fmt_ts(x) if pd.notna(x) else "")
metric_results['latest_anomaly_timestamp'] = metric_results['latest_anomaly_ts_num'].apply(lambda x: fmt_ts(x) if pd.notna(x) else "")

metric_results = metric_results[[
    'cmdb_id','kpi_name','global_count','global_p95',
    'window_row_count','anomaly_count_in_window',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window'
]]

metric_results = metric_results.sort_values(by=['anomaly_count_in_window','earliest_anomaly_timestamp'], ascending=[False,True]).head(20)
metric_results[['global_p95','max_value_in_window']] = metric_results[['global_p95','max_value_in_window']].round(6)

# Display the two compact result tables (CSV-like)
trace_results, metric_results

```
Out[3]:
```
```python
# Compose a plain-English summary string based on previous computation variables
try:
    trace_df = trace_results
    metric_df = metric_results
except NameError:
    # fallback: load results if not present (shouldn't be necessary in stateful kernel)
    import pandas as pd
    trace_df = pd.read_csv("trace_results.csv") if False else pd.DataFrame()
    metric_df = pd.read_csv("metric_results.csv") if False else pd.DataFrame()

lines = []
lines.append("Trace.csv P95-anomaly summary (window 2020-05-27 02:00–02:30 CST):")
if trace_df.empty:
    lines.append("- No matching trace anomalies found for the selected groups.")
else:
    for _, r in trace_df.iterrows():
        lines.append(
            f"- {r['cmdb_id']} / {r['trace_name']}: global_count={int(r['global_count'])}, "
            f"global_p95={r['global_p95']}, window_rows={int(r['window_row_count'])}, "
            f"anomalies_in_window={int(r['anomaly_count_in_window'])}, "
            f"earliest={r['earliest_anomaly_timestamp']}, latest={r['latest_anomaly_timestamp']}, "
            f"max_in_window={r['max_value_in_window']}"
        )

lines.append("")  # blank line
lines.append("Metric.csv tnsping_result_time summary for db_001/db_002/db_003 (P95-based):")
if metric_df.empty:
    lines.append("- No tnsping_result_time anomalies found for the selected DBs.")
else:
    for _, r in metric_df.iterrows():
        lines.append(
            f"- {r['cmdb_id']} / {r['kpi_name']}: global_count={int(r['global_count'])}, "
            f"global_p95={r['global_p95']}, window_rows={int(r['window_row_count'])}, "
            f"anomalies_in_window={int(r['anomaly_count_in_window'])}, "
            f"earliest={r['earliest_anomaly_timestamp']}, latest={r['latest_anomaly_timestamp']}, "
            f"max_in_window={r['max_value_in_window']}"
        )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                     trace_name  global_count  global_p95  window_row_count  anomaly_count_in_window earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window
1  docker_001   trace.from_os_022.remoteprocess.duration_p95            30    1.566297                30                        2    2020-05-27 02:06:00 CST  2020-05-27 02:08:00 CST             2.732500
0  docker_001  trace.from_os_022.remoteprocess.duration_mean            30    0.519370                30                        2    2020-05-27 02:08:00 CST  2020-05-27 02:09:00 CST             0.692284
2  docker_001                 trace.self.local.duration_mean            30    0.011995                30                        2    2020-05-27 02:08:00 CST  2020-05-27 02:09:00 CST             0.015122
3  docker_001                  trace.self.local.duration_p95            30    0.089550                30                        2    2020-05-27 02:08:00 CST  2020-05-27 02:09:00 CST             0.096000,   cmdb_id             kpi_name  global_count  global_p95  window_row_count  anomaly_count_in_window earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window
0  db_001  tnsping_result_time            48        10.0                30                       11    2020-05-27 02:03:00 CST  2020-05-27 02:22:00 CST                 10.0
2  db_003  tnsping_result_time            48        10.0                30                        9    2020-05-27 02:00:00 CST  2020-05-27 02:26:00 CST                 10.0
1  db_002  tnsping_result_time            48        10.0                30                        9    2020-05-27 02:02:00 CST  2020-05-27 02:29:00 CST                 10.0)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# Components and KPIs of interest
cmdbs = ['db_003','db_002','db_001','os_022','docker_001']
kpis = [
    'tnsping_result_time','ICMP_ping','Agent_ping','Send_total','Recv_total',
    'Incoming_network_traffic','Outgoing_network_traffic','CPU_free_pct','Proc_Used_Pct','MEM_Total'
]

# 1) Compute global P95 and P5 and global counts for relevant (cmdb_id,kpi_name) groups (using full series)
thresh_all = (
    df_metric[df_metric['cmdb_id'].isin(cmdbs) & df_metric['kpi_name'].isin(kpis)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
)
thresh_all[['global_p95','global_p5']] = thresh_all[['global_p95','global_p5']].round(6)

# 2) Filter to incident window (Asia/Shanghai 2020-05-27 02:00:00 through 02:30:00 inclusive)
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp()); end_ts = int(end_dt.timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Keep only rows for targeted cmdb/kpis
df_window_sel = df_window[df_window['cmdb_id'].isin(cmdbs) & df_window['kpi_name'].isin(kpis)].copy()

# 3) Define metric categories for anomaly direction
# Use >=P95 for "higher-is-bad" metrics (latency/volume/resource high usage)
higher_is_bad = {
    'tnsping_result_time','ICMP_ping','Agent_ping',
    'Send_total','Recv_total','Incoming_network_traffic','Outgoing_network_traffic',
    'Proc_Used_Pct','MEM_Total'
}
# Use <=P5 for "lower-is-bad" metrics (availability/free percent)
lower_is_bad = {'CPU_free_pct'}

# Merge global thresholds into window data (thresholds computed on full series)
merged = df_window_sel.merge(thresh_all, on=['cmdb_id','kpi_name'], how='left')

# If no threshold (NaN), those groups are not present in full series -> drop
merged = merged[merged['global_count'].notna()].copy()

# Special rule for tnsping_result_time: if global_p95 == 0 then any value > 0 is anomaly
def detect_anomaly(row):
    if pd.isna(row['value']) or pd.isna(row['global_p95']) or pd.isna(row['global_p5']):
        return False
    k = row['kpi_name']
    v = row['value']
    if k == 'tnsping_result_time':
        if row['global_p95'] == 0:
            return v > 0
        else:
            return v >= row['global_p95']
    if k in higher_is_bad:
        return v >= row['global_p95']
    if k in lower_is_bad:
        return v <= row['global_p5']
    # fallback: treat as higher-is-bad
    return v >= row['global_p95']

merged['is_anom'] = merged.apply(detect_anomaly, axis=1)

# 4) Aggregate per group and collect required fields
agg_window = (
    merged
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(window_row_count=('value','count'),
         anomaly_count=('is_anom','sum'),
         earliest_anom_ts_num=('timestamp', lambda s: s[merged.loc[s.index,'is_anom']].min() if len(s)>0 else pd.NA),
         latest_anom_ts_num=('timestamp', lambda s: s[merged.loc[s.index,'is_anom']].max() if len(s)>0 else pd.NA),
         max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
)

# Join with global thresholds/counts
result = thresh_all.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')

# Keep only groups with >=1 anomaly in window
result = result[result['anomaly_count'].fillna(0) >= 1].copy()

# Format timestamps to Asia/Shanghai
def fmt_ts(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_anomaly_timestamp'] = result['earliest_anom_ts_num'].apply(fmt_ts)
result['latest_anomaly_timestamp'] = result['latest_anom_ts_num'].apply(fmt_ts)

# Round numeric columns
for c in ['global_p95','global_p5','max_value_in_window','min_value_in_window']:
    if c in result.columns:
        result[c] = result[c].round(6)

# Select requested output columns
result_out = result[[
    'cmdb_id','kpi_name','global_count','global_p95','global_p5',
    'window_row_count','anomaly_count','earliest_anomaly_timestamp',
    'latest_anomaly_timestamp','max_value_in_window','min_value_in_window'
]]

# Sort and limit to top 20 rows (anomaly_count desc, earliest_anomaly_timestamp asc)
result_out_sorted = result_out.sort_values(
    by=['anomaly_count','earliest_anomaly_timestamp'],
    ascending=[False, True]
).head(20)

result_out_sorted

```
Out[4]:
```
```python
# Compose a concise plain-English summary of the anomaly findings using the prior results (result_out_sorted).
# This displays a compact textual summary only.

try:
    df = result_out_sorted.copy()
except NameError:
    # If variable not present, load previous computation (fallback - unlikely in stateful kernel)
    import pandas as pd
    df = pd.DataFrame()

def fmt(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        # show integers without decimals where appropriate
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.6f}"
    return str(val)

lines = []
lines.append("Summary of detected anomalies (window 2020-05-27 02:00–02:30 CST):")
if df.empty:
    lines.append("- No results available.")
else:
    # Iterate rows and produce one-line summary per group
    for _, r in df.iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        gc = fmt(r['global_count'])
        p95 = fmt(r['global_p95'])
        p5 = fmt(r['global_p5'])
        wrows = fmt(r['window_row_count'])
        anom = fmt(r['anomaly_count'])
        ear = r['earliest_anomaly_timestamp'] or ""
        lat = r['latest_anomaly_timestamp'] or ""
        vmax = fmt(r['max_value_in_window'])
        vmin = fmt(r['min_value_in_window'])
        lines.append(
            f"- {cmdb} / {kpi}: global_count={gc}, p95={p95}, p5={p5}, window_rows={wrows}, "
            f"anomalies={anom}, earliest={ear}, latest={lat}, max={vmax}, min={vmin}"
        )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name  global_count  global_p95   global_p5  window_row_count  anomaly_count earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  min_value_in_window
1   db_001                 MEM_Total            48  381.902264  381.902264                30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           381.902264           381.902264
5   db_002                 MEM_Total            48  381.902264  381.902264                30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           381.902264           381.902264
9   db_003                 MEM_Total            48  381.902264  381.902264                30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST           381.902264           381.902264
12  os_022                Agent_ping            48    1.000000    1.000000                30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.000000             1.000000
13  os_022                 ICMP_ping            48    1.000000    1.000000                30             30    2020-05-27 02:00:00 CST  2020-05-27 02:29:00 CST             1.000000             1.000000
2   db_001             Proc_Used_Pct            47    1.540000    1.530000                30             13    2020-05-27 02:14:00 CST  2020-05-27 02:29:00 CST             1.540000             1.530000
3   db_001       tnsping_result_time            48   10.000000    0.000000                30             11    2020-05-27 02:03:00 CST  2020-05-27 02:22:00 CST            10.000000             0.000000
11  db_003       tnsping_result_time            48   10.000000    0.000000                30              9    2020-05-27 02:00:00 CST  2020-05-27 02:26:00 CST            10.000000             0.000000
7   db_002       tnsping_result_time            48   10.000000    0.000000                30              9    2020-05-27 02:02:00 CST  2020-05-27 02:29:00 CST            10.000000             0.000000
4   db_002              CPU_free_pct            48   99.384384   98.690505                30              3    2020-05-27 02:00:00 CST  2020-05-27 02:15:00 CST            99.314514            98.543769
8   db_003              CPU_free_pct            49   98.975657   96.592927                31              3    2020-05-27 02:02:00 CST  2020-05-27 02:26:00 CST            99.046206            96.003210
6   db_002             Proc_Used_Pct            46    1.360000    1.342500                29              3    2020-05-27 02:23:00 CST  2020-05-27 02:25:00 CST             1.370000             1.340000
14  os_022  Incoming_network_traffic            48    0.045488    0.020335                30              2    2020-05-27 02:00:00 CST  2020-05-27 02:07:00 CST             0.050339             0.018609
15  os_022  Outgoing_network_traffic            49    0.073664    0.039791                31              2    2020-05-27 02:00:00 CST  2020-05-27 02:12:00 CST             0.081375             0.036406
0   db_001              CPU_free_pct            49   99.344713   98.182262                31              2    2020-05-27 02:02:00 CST  2020-05-27 02:12:00 CST            99.442595            98.060754
16  os_022                Recv_total            47   47.057757   20.435530                30              1    2020-05-27 02:11:00 CST  2020-05-27 02:11:00 CST            47.262073            19.046107
17  os_022                Send_total            47    0.074067    0.039764                30              1    2020-05-27 02:11:00 CST  2020-05-27 02:11:00 CST             0.074430             0.036612```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 and counts for each (cmdb_id, kpi_name) using full series
thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
)
thresh[['global_p95','global_p5']] = thresh[['global_p95','global_p5']].astype(float)

# 2) Filter to incident window (Asia/Shanghai 2020-05-27 02:00:00 through 2020-05-27 02:30:00 inclusive)
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window and detect anomalies with special tnsping rule
merged = df_window.merge(thresh, on=['cmdb_id','kpi_name'], how='left')

# Drop groups without thresholds (shouldn't usually happen)
merged = merged[merged['global_count'].notna()].copy()

def is_anomaly_row(r):
    v = r['value']
    p95 = r['global_p95']
    p5 = r['global_p5']
    k = r['kpi_name']
    # tnsping special-case
    if k == 'tnsping_result_time':
        if p95 == 0:
            return v > 0
        return v >= p95
    # default rule: anomaly if >= p95 or <= p5
    return (v >= p95) or (v <= p5)

merged['is_anom'] = merged.apply(is_anomaly_row, axis=1)

# Keep only anomaly rows in window
anom = merged[merged['is_anom']].copy()
if anom.empty:
    # create empty DataFrame with required columns
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_count','global_p95','global_p5',
        'earliest_anomaly_timestamp','earliest_value','absolute_deviation','relative_deviation_pct'
    ])
else:
    # For each group find earliest anomaly point (min timestamp) and its value
    idx_min = anom.groupby(['cmdb_id','kpi_name'])['timestamp'].idxmin()
    earliest_rows = anom.loc[idx_min].reset_index(drop=True)

    # Compute deviations
    gv95 = earliest_rows['global_p95'].values.astype(float)
    gv5 = earliest_rows['global_p5'].values.astype(float)
    ev = earliest_rows['value'].values.astype(float)

    # absolute_deviation: (earliest_value - global_p95) when earliest_value>=global_p95, else (global_p5 - earliest_value)
    abs_dev = np.where(ev >= gv95, ev - gv95, gv5 - ev)

    # relative_deviation_pct = absolute_deviation / (global_p95 if global_p95>0 else 1) * 100
    denom = np.where(gv95 > 0, gv95, 1.0)
    rel_pct = abs_dev / denom * 100.0

    # Build result DataFrame
    earliest_rows['earliest_value'] = ev
    earliest_rows['absolute_deviation'] = abs_dev
    earliest_rows['relative_deviation_pct'] = rel_pct

    # Format timestamp to Asia/Shanghai
    def fmt_ts_num(x):
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    earliest_rows['earliest_anomaly_timestamp'] = earliest_rows['timestamp'].apply(fmt_ts_num)

    result = earliest_rows[[
        'cmdb_id','kpi_name','global_count','global_p95','global_p5',
        'earliest_anomaly_timestamp','earliest_value','absolute_deviation','relative_deviation_pct'
    ]].copy()

    # Round numeric columns for compactness
    for c in ['global_p95','global_p5','earliest_value','absolute_deviation','relative_deviation_pct']:
        result[c] = result[c].astype(float).round(6)

    # 4) Sort by earliest_anomaly_timestamp ascending and limit to top 20
    result['__ts_sort'] = pd.to_datetime(result['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
    result_sorted = result.sort_values(by='__ts_sort', ascending=True).drop(columns='__ts_sort').head(20)

    result_top20 = result_sorted

# Display compact CSV-like table (DataFrame variable as per rules)
result_top20

```
Out[5]:
```
```python
# Compose a concise plain-English summary string from the previously computed result_top20 DataFrame.
# The summary lists how many groups had earliest anomalies and shows up to the first 10 groups with key details.
try:
    df = result_top20.copy()
except NameError:
    # If result_top20 is not present, load minimal data to avoid failure (fallback)
    import pandas as pd
    df = pd.DataFrame()

if df.empty:
    summary = "No anomalies detected in the incident window or no data available."
else:
    total_groups = df.shape[0]
    lines = [f"Anomaly earliest-point summary: {total_groups} groups detected with >=1 anomaly in window."]
    lines.append("Showing up to first 10 groups ordered by earliest_anomaly_timestamp:")
    for _, r in df.sort_values('earliest_anomaly_timestamp').head(10).iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name'] or ""
        ts = r['earliest_anomaly_timestamp'] or ""
        ev = float(r['earliest_value']) if pd.notna(r['earliest_value']) else None
        absd = float(r['absolute_deviation']) if pd.notna(r['absolute_deviation']) else None
        rel = float(r['relative_deviation_pct']) if pd.notna(r['relative_deviation_pct']) else None
        gp95 = float(r['global_p95']) if 'global_p95' in r and pd.notna(r['global_p95']) else None
        gp5  = float(r['global_p5'])  if 'global_p5'  in r and pd.notna(r['global_p5'])  else None
        lines.append(
            f"- {cmdb} / {kpi}: earliest={ts}, earliest_value={ev:.6f}, "
            f"global_p95={gp95:.6f}, global_p5={gp5:.6f}, abs_dev={absd:.6f}, rel_dev%={rel:.2f}"
        )
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name  global_count     global_p95      global_p5 earliest_anomaly_timestamp  earliest_value  absolute_deviation  relative_deviation_pct
0     db_001                        ACS            47       0.020000       0.010000    2020-05-27 02:00:00 CST        0.010000            0.000000                0.000000
1044  os_009                Disk_wr_ios            48       2.000000       1.000000    2020-05-27 02:00:00 CST        2.000000            0.000000                0.000000
1047  os_009                FS_max_util            10      61.000000      61.000000    2020-05-27 02:00:00 CST       61.000000            0.000000                0.000000
1048  os_009             FS_total_space            10      58.768400      58.768400    2020-05-27 02:00:00 CST       58.768400            0.000000                0.000000
1051  os_009                  ICMP_ping            48       1.000000       1.000000    2020-05-27 02:00:00 CST        1.000000            0.000000                0.000000
1059  os_009   Num_of_running_processes            10       2.000000       2.000000    2020-05-27 02:00:00 CST        2.000000            0.000000                0.000000
1061  os_009                    Page_pi            48       0.000000       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1063  os_009      Processor_load_15_min            48       0.000833       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1064  os_009       Processor_load_1_min            48       0.009417       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1068  os_009             Received_queue            10       0.000000       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1073  os_009                 Sent_queue            10     189.100000       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1074  os_009              Shared_memory            10  707963.400000  707948.000000    2020-05-27 02:00:00 CST   707948.000000            0.000000                0.000000
1076  os_009  System_block_queue_length            10       0.000000       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1078  os_009             Zombie_Process            10       0.000000       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1080  os_010                 Agent_ping            48       1.000000       1.000000    2020-05-27 02:00:00 CST        1.000000            0.000000                0.000000
1082  os_010               CPU_idle_pct            48      99.908584      99.876868    2020-05-27 02:00:00 CST       99.871257            0.005611                0.005616
1083  os_010            CPU_iowait_time            48       0.000926       0.000000    2020-05-27 02:00:00 CST        0.000000            0.000000                0.000000
1085  os_010              CPU_user_time            47       0.068262       0.047516    2020-05-27 02:00:00 CST        0.072250            0.003988                5.841886
1111  os_010   Outgoing_network_traffic            48       0.000920       0.000267    2020-05-27 02:00:00 CST        0.001244            0.000324               35.158627
1108  os_010            Memory_used_pct            48       7.520000       7.500000    2020-05-27 02:00:00 CST        7.500000            0.000000                0.000000```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing loaded dataframes if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# Components of interest
components = ['db_003','db_002','db_001','os_022','docker_001']

# Incident window in UTC (convert from Asia/Shanghai)
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp()); end_ts = int(end_dt.timestamp())

# 1) Global thresholds for metrics (full series)
metric_thresh = (
    df_metric
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
)
metric_thresh['global_p95'] = metric_thresh['global_p95'].astype(float)
metric_thresh['global_p5'] = metric_thresh['global_p5'].astype(float)

# 1b) Global P95 for traces (full series)
trace_thresh = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95))
)
trace_thresh['global_p95'] = trace_thresh['global_p95'].astype(float)

# 2) Filter both datasets to the incident window
df_metric_w = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_trace_w  = df_trace[(df_trace['timestamp']  >= start_ts) & (df_trace['timestamp']  <= end_ts)].copy()

# Keep only components of interest in window
df_metric_w = df_metric_w[df_metric_w['cmdb_id'].isin(components)].copy()
df_trace_w  = df_trace_w[df_trace_w['cmdb_id'].isin(components)].copy()

# 3) Detect anomalies
# Merge thresholds into window metric data
merged_metric = df_metric_w.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')
# Drop rows without thresholds (no global data)
merged_metric = merged_metric[merged_metric['global_count'].notna()].copy()

def metric_is_anom(r):
    v = r['value']
    p95 = r['global_p95']
    p5 = r['global_p5']
    k = r['kpi_name']
    # tnsping special-case
    if k == 'tnsping_result_time':
        if p95 == 0:
            return v > 0
        return v >= p95
    return (v >= p95) or (v <= p5)

merged_metric['is_anom'] = merged_metric.apply(metric_is_anom, axis=1)

# Merge thresholds into window trace data
merged_trace = df_trace_w.merge(trace_thresh, left_on=['cmdb_id','trace_name'], right_on=['cmdb_id','trace_name'], how='left')
merged_trace = merged_trace[merged_trace['global_count'].notna()].copy()
# trace anomaly rule: value >= global_p95
merged_trace['is_anom'] = merged_trace['value'] >= merged_trace['global_p95']

# 4) For each component find earliest anomaly point across metrics and traces
# Prepare metric anomaly rows with unified schema
metric_anoms = merged_metric[merged_metric['is_anom']].copy()
if not metric_anoms.empty:
    metric_anoms = metric_anoms.assign(
        source_type='metric',
        kpi_or_trace_name=metric_anoms['kpi_name'],
        global_p95_metric=metric_anoms['global_p95'],
        global_p5_metric=metric_anoms['global_p5'],
        global_p95_trace=np.nan  # placeholder
    )
    metric_anoms = metric_anoms[[
        'cmdb_id','source_type','kpi_or_trace_name','timestamp','value',
        'global_p95_metric','global_p5_metric'
    ]]
else:
    metric_anoms = pd.DataFrame(columns=['cmdb_id','source_type','kpi_or_trace_name','timestamp','value','global_p95_metric','global_p5_metric'])

# Prepare trace anomaly rows with unified schema
trace_anoms = merged_trace[merged_trace['is_anom']].copy()
if not trace_anoms.empty:
    trace_anoms = trace_anoms.assign(
        source_type='trace',
        kpi_or_trace_name=trace_anoms['trace_name'],
        global_p95_metric=np.nan,
        global_p5_metric=np.nan
    )
    trace_anoms = trace_anoms[[
        'cmdb_id','source_type','kpi_or_trace_name','timestamp','value',
        'global_p95','global_count'
    ]].rename(columns={'global_p95':'global_p95_metric','global_count':'_gc'})
    # keep global_p95_metric as the trace p95; global_p5_metric stays NaN
    trace_anoms['global_p5_metric'] = np.nan
    trace_anoms = trace_anoms[['cmdb_id','source_type','kpi_or_trace_name','timestamp','value','global_p95_metric','global_p5_metric']]
else:
    trace_anoms = pd.DataFrame(columns=['cmdb_id','source_type','kpi_or_trace_name','timestamp','value','global_p95_metric','global_p5_metric'])

# Combine anomalies
combined = pd.concat([metric_anoms, trace_anoms], ignore_index=True, sort=False)

# If no anomalies at all, return empty DataFrame with requested columns
if combined.empty:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','source_type','kpi_or_trace_name','earliest_anomaly_timestamp',
        'earliest_value','global_p95','global_p5','category',
        'anomaly_kpi_count','total_anomaly_points'
    ])
else:
    # 5) Compute anomaly_kpi_count and total_anomaly_points per component (distinct kpi_or_trace_name and total rows)
    agg_counts = combined.groupby('cmdb_id').agg(
        anomaly_kpi_count=('kpi_or_trace_name', lambda s: s.nunique()),
        total_anomaly_points=('kpi_or_trace_name', 'count')
    ).reset_index()

    # For each component find earliest anomaly row (min timestamp)
    idx_earliest = combined.groupby('cmdb_id')['timestamp'].idxmin()
    earliest_rows = combined.loc[idx_earliest].reset_index(drop=True)

    # Category labeling function
    def category_label(name, source_type):
        n = (name or "").lower()
        if source_type == 'trace' or n.startswith('trace.') or 'trace' in n:
            return 'trace'
        if 'tnsping' in n or 'tns' in n:
            return 'tnsping'
        net_keywords = ['icmp','agent_ping','send','recv','incoming_network','outgoing_network','network','ping']
        for kw in net_keywords:
            if kw in n:
                return 'network'
        res_keywords = ['cpu','proc_','mem','memory','used_pct','used','cpu_free','proc_used','procuser']
        for kw in res_keywords:
            if kw in n:
                return 'resource'
        return 'other'

    earliest_rows['category'] = earliest_rows.apply(lambda r: category_label(r['kpi_or_trace_name'], r['source_type']), axis=1)

    # Format timestamp to Asia/Shanghai
    def fmt_ts_num(x):
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    earliest_rows['earliest_anomaly_timestamp'] = earliest_rows['timestamp'].apply(fmt_ts_num)
    earliest_rows['earliest_value'] = earliest_rows['value'].round(6)
    # global_p95 (for metrics from metric_anoms use global_p95_metric; for traces this holds trace p95)
    earliest_rows['global_p95'] = earliest_rows['global_p95_metric'].round(6)
    # global_p5: for metrics we have values in metric_anoms; for traces leave blank (NaN)
    # For metric_anoms we used column global_p5_metric; ensure present
    if 'global_p5_metric' in earliest_rows.columns:
        earliest_rows['global_p5'] = earliest_rows['global_p5_metric'].round(6)
    else:
        earliest_rows['global_p5'] = np.nan

    # Join counts
    result = earliest_rows.merge(agg_counts, on='cmdb_id', how='left')

    # Select and order columns as requested
    result_df = result[[
        'cmdb_id','source_type','kpi_or_trace_name','earliest_anomaly_timestamp',
        'earliest_value','global_p95','global_p5','category',
        'anomaly_kpi_count','total_anomaly_points'
    ]].copy()

    # Sort by earliest_anomaly_timestamp ascending and limit to 10 rows
    result_df['__ts_sort'] = pd.to_datetime(result_df['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
    result_df = result_df.sort_values('__ts_sort', ascending=True).drop(columns='__ts_sort').head(10)

# Final compact output
result_df

```
Out[6]:
```
db_001 — metric "ACS" at 2020-05-27 02:00:00 CST; earliest_value=0.010000; global_p95=0.020000; global_p5=0.010000; category=other; anomaly_kpi_count=43; total_anomaly_points=635

db_002 — metric "ACS" at 2020-05-27 02:00:00 CST; earliest_value=0.020000; global_p95=0.020000; global_p5=0.013000; category=other; anomaly_kpi_count=43; total_anomaly_points=520

db_003 — metric "ACS" at 2020-05-27 02:00:00 CST; earliest_value=0.020000; global_p95=0.020000; global_p5=0.003000; category=other; anomaly_kpi_count=42; total_anomaly_points=362

docker_001 — metric "container_fgc" at 2020-05-27 02:00:00 CST; earliest_value=0.000000; global_p95=0.000000; global_p5=0.000000; category=other; anomaly_kpi_count=37; total_anomaly_points=494

os_022 — metric "Agent_ping" at 2020-05-27 02:00:00 CST; earliest_value=1.000000; global_p95=1.000000; global_p5=1.000000; category=network; anomaly_kpi_count=67; total_anomaly_points=560

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id source_type kpi_or_trace_name earliest_anomaly_timestamp  earliest_value  global_p95  global_p5 category  anomaly_kpi_count  total_anomaly_points
0      db_001      metric               ACS    2020-05-27 02:00:00 CST            0.01        0.02      0.010    other                 43                   635
1      db_002      metric               ACS    2020-05-27 02:00:00 CST            0.02        0.02      0.013    other                 43                   520
2      db_003      metric               ACS    2020-05-27 02:00:00 CST            0.02        0.02      0.003    other                 42                   362
3  docker_001      metric     container_fgc    2020-05-27 02:00:00 CST            0.00        0.00      0.000    other                 37                   494
4      os_022      metric        Agent_ping    2020-05-27 02:00:00 CST            1.00        1.00      1.000  network                 67                   560```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing dfs if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# Define targets
tns_dbs = ['db_003','db_002','db_001']
tns_kpi = 'tnsping_result_time'
os022_kpis = ['Agent_ping','ICMP_ping','Incoming_network_traffic','Outgoing_network_traffic']
docker_traces = [
    ('docker_001','trace.from_os_022.remoteprocess.duration_mean'),
    ('docker_001','trace.from_os_022.remoteprocess.duration_p95'),
]

# Incident window (Asia/Shanghai -> convert to UTC timestamps)
start_dt = pd.Timestamp("2020-05-27 02:00:00", tz=tz).tz_convert('UTC')
end_dt   = pd.Timestamp("2020-05-27 02:30:00", tz=tz).tz_convert('UTC')
start_ts = int(start_dt.timestamp()); end_ts = int(end_dt.timestamp())

# 1) Compute global thresholds using full series (metrics & traces) for only the target groups
metric_targets = []
for db in tns_dbs:
    metric_targets.append((db, tns_kpi))
for k in os022_kpis:
    metric_targets.append(('os_022', k))

metric_targets_df = pd.DataFrame(metric_targets, columns=['cmdb_id','kpi_name'])

# Filter full metric series to only target groups (thresholds computed on full series for these groups)
metric_full_sel = df_metric.merge(metric_targets_df, on=['cmdb_id','kpi_name'], how='inner')

metric_thresh = (
    metric_full_sel
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
)
metric_thresh[['global_p95','global_p5']] = metric_thresh[['global_p95','global_p5']].astype(float)

# Trace thresholds for docker_001 traces
trace_targets_df = pd.DataFrame(docker_traces, columns=['cmdb_id','trace_name'])
trace_full_sel = df_trace.merge(trace_targets_df, on=['cmdb_id','trace_name'], how='inner')
trace_thresh = (
    trace_full_sel
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_count='count',
         global_p95=lambda s: s.quantile(0.95))
)
trace_thresh['global_p95'] = trace_thresh['global_p95'].astype(float)

# 2) Filter data to incident window
metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
metric_window = metric_window.merge(metric_targets_df, on=['cmdb_id','kpi_name'], how='inner')

trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
trace_window = trace_window.merge(trace_targets_df, on=['cmdb_id','trace_name'], how='inner')

# 3) Detect anomalies
# Merge thresholds into window rows
metric_w = metric_window.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')
# Note: metric_thresh computed before filtering as required

def metric_is_anom(r):
    v = r['value']
    p95 = r['global_p95']
    p5 = r['global_p5']
    k = r['kpi_name']
    if pd.isna(p95) or pd.isna(p5):
        return False
    if k == 'tnsping_result_time':
        if p95 == 0:
            return v > 0
        return v >= p95
    return (v >= p95) or (v <= p5)

metric_w['is_anom'] = metric_w.apply(metric_is_anom, axis=1)

# Traces: merge thresholds and detect value >= global_p95
trace_w = trace_window.merge(trace_thresh, on=['cmdb_id','trace_name'], how='left')
trace_w['is_anom'] = trace_w.apply(lambda r: False if pd.isna(r['global_p95']) else (r['value'] >= r['global_p95']), axis=1)

# 4) For each target produce one row (include groups even if 0 anomalies)
rows = []

# Helper to format timestamp
def fmt_ts_num(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Process metric targets
for cmdb,kpi in metric_targets:
    grp_all = metric_thresh[(metric_thresh['cmdb_id']==cmdb)&(metric_thresh['kpi_name']==kpi)]
    if not grp_all.empty:
        gc = int(grp_all.iloc[0]['global_count'])
        gp95 = float(grp_all.iloc[0]['global_p95'])
        gp5  = float(grp_all.iloc[0]['global_p5'])
    else:
        gc = 0; gp95 = np.nan; gp5 = np.nan
    # window rows
    wrows = metric_w[(metric_w['cmdb_id']==cmdb)&(metric_w['kpi_name']==kpi)].copy()
    window_row_count = int(wrows.shape[0])
    anomaly_rows = wrows[wrows['is_anom']].copy()
    anomaly_count = int(anomaly_rows.shape[0])
    if anomaly_count>0:
        idx = anomaly_rows['timestamp'].idxmin()
        earliest_ts_num = int(anomaly_rows.loc[idx,'timestamp'])
        earliest_value = float(anomaly_rows.loc[idx,'value'])
        max_value = float(wrows['value'].max()) if window_row_count>0 else np.nan
        earliest_ts = fmt_ts_num(earliest_ts_num)
    else:
        earliest_ts = ""
        earliest_value = np.nan
        max_value = float(wrows['value'].max()) if window_row_count>0 else np.nan
    rows.append({
        'cmdb_id': cmdb,
        'metric_or_trace_name': kpi,
        'data_source': 'metric',
        'global_count': gc,
        'global_p95': round(gp95,6) if not np.isnan(gp95) else np.nan,
        'global_p5': round(gp5,6) if not np.isnan(gp5) else np.nan,
        'window_row_count': window_row_count,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'earliest_value_in_window': round(earliest_value,6) if not np.isnan(earliest_value) else np.nan,
        'max_value_in_window': round(max_value,6) if not np.isnan(max_value) else np.nan
    })

# Process os_022 targets already included above if included; ensure no duplicate - metric_targets included os_022 set.

# Process trace targets for docker_001
for cmdb,trace_name in docker_traces:
    grp_all = trace_thresh[(trace_thresh['cmdb_id']==cmdb)&(trace_thresh['trace_name']==trace_name)]
    if not grp_all.empty:
        gc = int(grp_all.iloc[0]['global_count'])
        gp95 = float(grp_all.iloc[0]['global_p95'])
    else:
        gc = 0; gp95 = np.nan
    wrows = trace_w[(trace_w['cmdb_id']==cmdb)&(trace_w['trace_name']==trace_name)].copy()
    window_row_count = int(wrows.shape[0])
    anomaly_rows = wrows[wrows['is_anom']].copy()
    anomaly_count = int(anomaly_rows.shape[0])
    if anomaly_count>0:
        idx = anomaly_rows['timestamp'].idxmin()
        earliest_ts_num = int(anomaly_rows.loc[idx,'timestamp'])
        earliest_value = float(anomaly_rows.loc[idx,'value'])
        max_value = float(wrows['value'].max()) if window_row_count>0 else np.nan
        earliest_ts = fmt_ts_num(earliest_ts_num)
    else:
        earliest_ts = ""
        earliest_value = np.nan
        max_value = float(wrows['value'].max()) if window_row_count>0 else np.nan
    rows.append({
        'cmdb_id': cmdb,
        'metric_or_trace_name': trace_name,
        'data_source': 'trace',
        'global_count': gc,
        'global_p95': round(gp95,6) if not np.isnan(gp95) else np.nan,
        'global_p5': "",  # blank for traces
        'window_row_count': window_row_count,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'earliest_value_in_window': round(earliest_value,6) if not np.isnan(earliest_value) else np.nan,
        'max_value_in_window': round(max_value,6) if not np.isnan(max_value) else np.nan
    })

# Assemble result DataFrame
result = pd.DataFrame(rows)

# Sort by earliest_anomaly_timestamp ascending (empty timestamps go last)
result['__ts_sort'] = pd.to_datetime(result['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
result_sorted = result.sort_values(by='__ts_sort', ascending=True).drop(columns='__ts_sort').head(20)

# Final compact table
result_sorted

```
Out[7]:
```
Summary of earliest anomalies in the incident window (2020-05-27 02:00–02:30 CST):

- db_003 / tnsping_result_time (metric): global_count=48, p95=10.000000, p5=0.0; window_rows=30, anomalies=9; earliest=2020-05-27 02:00:00 CST, earliest_value=10.000000, max_in_window=10.000000

- os_022 / Agent_ping (metric): global_count=48, p95=1.000000, p5=1.0; window_rows=30, anomalies=30; earliest=2020-05-27 02:00:00 CST, earliest_value=1.000000, max_in_window=1.000000

- os_022 / ICMP_ping (metric): global_count=48, p95=1.000000, p5=1.0; window_rows=30, anomalies=30; earliest=2020-05-27 02:00:00 CST, earliest_value=1.000000, max_in_window=1.000000

- os_022 / Incoming_network_traffic (metric): global_count=48, p95=0.045488, p5=0.020335; window_rows=30, anomalies=5; earliest=2020-05-27 02:00:00 CST, earliest_value=0.050339, max_in_window=0.050339

- os_022 / Outgoing_network_traffic (metric): global_count=49, p95=0.073664, p5=0.039791; window_rows=31, anomalies=5; earliest=2020-05-27 02:00:00 CST, earliest_value=0.081375, max_in_window=0.081375

- db_002 / tnsping_result_time (metric): global_count=48, p95=10.000000, p5=0.0; window_rows=30, anomalies=9; earliest=2020-05-27 02:02:00 CST, earliest_value=10.000000, max_in_window=10.000000

- db_001 / tnsping_result_time (metric): global_count=48, p95=10.000000, p5=0.0; window_rows=30, anomalies=11; earliest=2020-05-27 02:03:00 CST, earliest_value=10.000000, max_in_window=10.000000

- docker_001 / trace.from_os_022.remoteprocess.duration_p95 (trace): global_count=30, p95=1.566297; window_rows=30, anomalies=2; earliest=2020-05-27 02:06:00 CST, earliest_value=1.688000, max_in_window=2.732500

- docker_001 / trace.from_os_022.remoteprocess.duration_mean (trace): global_count=30, p95=0.519370; window_rows=30, anomalies=2; earliest=2020-05-27 02:08:00 CST, earliest_value=0.692284, max_in_window=0.692284

Notes: tnsping_result_time used the special rule (if global P95==0 then any value>0 is anomalous); trace anomalies use value >= global P95.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           metric_or_trace_name data_source  global_count  global_p95 global_p5  window_row_count  anomaly_count_in_window earliest_anomaly_timestamp  earliest_value_in_window  max_value_in_window
0      db_003                            tnsping_result_time      metric            48   10.000000       0.0                30                        9    2020-05-27 02:00:00 CST                 10.000000            10.000000
3      os_022                                     Agent_ping      metric            48    1.000000       1.0                30                       30    2020-05-27 02:00:00 CST                  1.000000             1.000000
4      os_022                                      ICMP_ping      metric            48    1.000000       1.0                30                       30    2020-05-27 02:00:00 CST                  1.000000             1.000000
5      os_022                       Incoming_network_traffic      metric            48    0.045488  0.020335                30                        5    2020-05-27 02:00:00 CST                  0.050339             0.050339
6      os_022                       Outgoing_network_traffic      metric            49    0.073664  0.039791                31                        5    2020-05-27 02:00:00 CST                  0.081375             0.081375
1      db_002                            tnsping_result_time      metric            48   10.000000       0.0                30                        9    2020-05-27 02:02:00 CST                 10.000000            10.000000
2      db_001                            tnsping_result_time      metric            48   10.000000       0.0                30                       11    2020-05-27 02:03:00 CST                 10.000000            10.000000
8  docker_001   trace.from_os_022.remoteprocess.duration_p95       trace            30    1.566297                          30                        2    2020-05-27 02:06:00 CST                  1.688000             2.732500
7  docker_001  trace.from_os_022.remoteprocess.duration_mean       trace            30    0.519370                          30                        2    2020-05-27 02:08:00 CST                  0.692284             0.692284```
```